# SynBioCrow 2.3 — Galaxy Development Evaluation V4\nMemory-safe offline evaluation. Uses a lightweight SMILES-only ensemble graph, processes one target at a time, checkpoints every target to Drive, and reports strict-stereo and connectivity-level recovery. No retrosynthesis backend is rerun.\n

In [ ]:
import os,sys,subprocess,shutil,json,zipfile,hashlib,time
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_galaxy_eval_v4")
SEALED=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_development_sealed")
NORM=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_literature_normalized")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_development_evaluation_v4")
OUT.mkdir(parents=True,exist_ok=True)

def run_stream(cmd,*,cwd=None,timeout=1800):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.Popen(list(map(str,cmd)),cwd=cwd,text=True,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    t0=time.time(); lines=[]
    for line in p.stdout:
        print(line.rstrip(),flush=True); lines.append(line)
        if time.time()-t0>timeout:
            p.kill(); raise subprocess.TimeoutExpired(cmd,timeout)
    rc=p.wait()
    if rc: raise subprocess.CalledProcessError(rc,cmd,output="".join(lines))
    return rc

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","--branch","develop/2.3",
                "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],check=True)

pred=SEALED/"sealed_predictions.json"
bench=NORM/"galaxy_literature_benchmark_normalized.json"
manifest=json.loads((SEALED/"sealed_manifest.json").read_text())
actual=hashlib.sha256(pred.read_bytes()).hexdigest()
assert actual==manifest["predictions_sha256"],(actual,manifest["predictions_sha256"])
print("SEALED PREDICTIONS VERIFIED",actual)

run_stream([sys.executable,str(REPO/"scripts/evaluate_galaxy_development_v4_2_3.py"),
            "--predictions",str(pred),"--benchmark",str(bench),"--output-dir",str(OUT)],
           cwd=REPO,timeout=1800)

bundle=OUT/"SynBioCrow_2_3_GALAXY_DEVELOPMENT_EVALUATION_V4.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*.json")):
        z.write(p,str(p.relative_to(OUT)))
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
